# RubyGuardian ML Classifier - Data Exploration

This notebook explores the Ruby script dataset used for malware classification.
We analyze feature distributions, class balance, and correlations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

DATA_DIR = Path('../data')
print(f'Data directory: {DATA_DIR.resolve()}')

In [ ]:
# Load extracted features
features_path = DATA_DIR / 'extracted_features.csv'
if features_path.exists():
    df = pd.read_csv(features_path)
    print(f'Dataset shape: {df.shape}')
    print(f'Columns: {list(df.columns)}')
    print(f'\nClass distribution:')
    print(df['label'].value_counts())
else:
    print('Feature file not found. Run feature extraction first.')
    # Create sample data for demonstration
    np.random.seed(42)
    n = 500
    df = pd.DataFrame({
        'eval_count': np.random.poisson(2, n),
        'base64_usage': np.random.binomial(1, 0.3, n),
        'network_calls': np.random.poisson(1, n),
        'file_operations': np.random.poisson(3, n),
        'obfuscation_score': np.random.beta(2, 5, n),
        'entropy': np.random.normal(4.5, 0.8, n),
        'line_count': np.random.lognormal(3, 1, n).astype(int),
        'dangerous_method_count': np.random.poisson(1.5, n),
        'string_concat_ratio': np.random.beta(2, 8, n),
        'label': np.random.choice(['benign', 'malicious', 'suspicious'], n, p=[0.5, 0.3, 0.2])
    })
    print(f'Using sample data: {df.shape}')

In [ ]:
# Class distribution visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['label'].value_counts().plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c', '#f39c12'])
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')

df['label'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c', '#f39c12'])
axes[1].set_title('Class Proportions')

plt.tight_layout()
plt.show()

In [ ]:
# Feature correlation heatmap
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr = df[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='RdBu_r', center=0, fmt='.2f')
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Feature distributions by class
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
features_to_plot = ['eval_count', 'obfuscation_score', 'entropy', 'network_calls', 'dangerous_method_count', 'string_concat_ratio']

for idx, feat in enumerate(features_to_plot):
    ax = axes[idx // 3][idx % 3]
    for label in df['label'].unique():
        subset = df[df['label'] == label]
        ax.hist(subset[feat], alpha=0.5, label=label, bins=20)
    ax.set_title(feat)
    ax.legend()

plt.suptitle('Feature Distributions by Class', fontsize=14)
plt.tight_layout()
plt.show()